# Creating Custom Tools

Every built-in SynapseKit tool is a `BaseTool` subclass with a `name`, `description`, and async `run()` method. Once you understand that contract, adding your own tools takes fewer than ten lines of code.

**What you'll build:** four increasingly capable tools — a one-liner decorator tool, a full `BaseTool` subclass with JSON Schema validation, an async HTTP tool, and a tool that returns structured data.

**Time:** ~20 min | **Difficulty:** Intermediate

**What you'll learn:**
- The `@tool` decorator for wrapping plain functions
- `BaseTool` subclassing for full control over schema and validation
- How `ToolResult` signals success vs error to the agent
- Async tools that call external APIs safely
- How the agent reads `tool.name` and `tool.description` at runtime

## Prerequisites

You need:
- An **OpenAI API key** (`OPENAI_API_KEY`)
- The `synapsekit` package (installed in the next cell)

In [ ]:
!pip install synapsekit -q

In [ ]:
import os

# Set your API key here before running agent cells
# os.environ['OPENAI_API_KEY'] = 'sk-...'

## Step 1: The @tool decorator (simplest path)

The `@tool` decorator wraps any sync or async function into a `BaseTool`. It infers the tool name from the function name and the description from the docstring. Type annotations become the JSON Schema for the tool's parameters.

In [ ]:
from synapsekit.agents import tool

@tool()
def celsius_to_fahrenheit(celsius: float) -> str:
    """Convert a temperature from Celsius to Fahrenheit."""
    return str(celsius * 9 / 5 + 32)

# The decorator returns a BaseTool instance, not the raw function
print(celsius_to_fahrenheit.name)         # "celsius_to_fahrenheit"
print(celsius_to_fahrenheit.description)  # "Convert a temperature..."

In [ ]:
# Override name and description explicitly when the function name would be ambiguous
@tool(name="unit_converter", description="Convert units of measurement. Supports temperature, length, and weight.")
def convert(value: float, from_unit: str, to_unit: str) -> str:
    """Convert between measurement units."""
    conversions = {
        ("celsius", "fahrenheit"): lambda v: v * 9 / 5 + 32,
        ("fahrenheit", "celsius"): lambda v: (v - 32) * 5 / 9,
        ("km", "miles"): lambda v: v * 0.621371,
        ("miles", "km"): lambda v: v / 0.621371,
    }
    key = (from_unit.lower(), to_unit.lower())
    fn = conversions.get(key)
    if fn is None:
        return f"Unsupported conversion: {from_unit} -> {to_unit}"
    return f"{fn(value):.4f} {to_unit}"

print(convert.name)        # "unit_converter"
print(convert.description) # "Convert units of measurement..."

## Step 2: BaseTool subclass for full schema control

The `@tool` decorator infers a basic schema. When you need enum values, pattern constraints, nested objects, or `additionalProperties: false`, subclass `BaseTool` directly and define `parameters` as a class attribute.

In [ ]:
import asyncio
from typing import Any
from synapsekit.agents import BaseTool, ToolResult

class StockPriceTool(BaseTool):
    """Fetch the current stock price for a given ticker symbol."""

    name = "stock_price"
    description = (
        "Look up the current price of a stock by its ticker symbol. "
        "Use for questions about stock prices, market cap, or trading volume."
    )
    parameters = {
        "type": "object",
        "properties": {
            "ticker": {
                "type": "string",
                "description": "Stock ticker symbol, e.g. AAPL, MSFT, GOOG",
                "pattern": "^[A-Z]{1,5}$",
            }
        },
        "required": ["ticker"],
        "additionalProperties": False,
    }

    async def run(self, ticker: str = "", **kwargs: Any) -> ToolResult:
        # Always validate inputs even though the schema already constrains them
        if not ticker:
            return ToolResult(output="", error="ticker is required")
        if not ticker.isalpha() or not ticker.isupper():
            return ToolResult(output="", error=f"Invalid ticker format: {ticker!r}")

        # Simulate a real API call; replace with httpx/aiohttp in production
        mock_prices = {"AAPL": 189.45, "MSFT": 415.20, "GOOG": 175.30}
        price = mock_prices.get(ticker)
        if price is None:
            return ToolResult(output="", error=f"Ticker not found: {ticker}")

        return ToolResult(output=f"{ticker}: ${price:.2f}")

# Quick test
tool_instance = StockPriceTool()
result = asyncio.run(tool_instance.run(ticker="AAPL"))
print(result.output)  # AAPL: $189.45

## Step 3: Async tools that call external APIs

Use `asyncio.get_running_loop().run_in_executor` to wrap any blocking I/O, or use `httpx.AsyncClient` directly. Either way, the tool stays non-blocking so the agent event loop is never stalled.

In [ ]:
import asyncio
from typing import Any
from synapsekit.agents import BaseTool, ToolResult

class ExchangeRateTool(BaseTool):
    """Fetch live currency exchange rates."""

    name = "exchange_rate"
    description = (
        "Get the current exchange rate between two currencies. "
        "Use ISO 4217 currency codes like USD, EUR, GBP, JPY."
    )
    parameters = {
        "type": "object",
        "properties": {
            "base": {"type": "string", "description": "Source currency code"},
            "target": {"type": "string", "description": "Target currency code"},
        },
        "required": ["base", "target"],
    }

    async def run(self, base: str = "", target: str = "", **kwargs: Any) -> ToolResult:
        base = (base or "").upper()
        target = (target or "").upper()

        if not base or not target:
            return ToolResult(output="", error="Both base and target currencies are required")

        # run_in_executor keeps blocking I/O off the event loop
        loop = asyncio.get_running_loop()
        try:
            result = await loop.run_in_executor(None, self._fetch_rate, base, target)
            return result
        except Exception as e:
            return ToolResult(output="", error=f"Failed to fetch rate: {e}")

    def _fetch_rate(self, base: str, target: str) -> ToolResult:
        # Simulated synchronous HTTP call; swap for requests.get() in production
        mock = {("USD", "EUR"): 0.92, ("EUR", "USD"): 1.09, ("USD", "GBP"): 0.79}
        rate = mock.get((base, target))
        if rate is None:
            return ToolResult(output="", error=f"No rate available for {base}/{target}")
        return ToolResult(output=f"1 {base} = {rate:.4f} {target}")

# Quick test
er_tool = ExchangeRateTool()
result = asyncio.run(er_tool.run(base="USD", target="EUR"))
print(result.output)  # 1 USD = 0.9200 EUR

## Step 4: Tools that return structured JSON

When downstream code needs to parse tool output, return JSON-encoded data inside `ToolResult.output`. The agent sees it as a string; your application can `json.loads()` it after the run.

In [ ]:
import json
from typing import Any
from synapsekit.agents import BaseTool, ToolResult

class CompanyInfoTool(BaseTool):
    """Return structured company information as JSON."""

    name = "company_info"
    description = (
        "Return structured information about a company: name, industry, "
        "founding year, and headquarters. Use for company research questions."
    )
    parameters = {
        "type": "object",
        "properties": {
            "company": {"type": "string", "description": "Company name or ticker"}
        },
        "required": ["company"],
    }

    async def run(self, company: str = "", **kwargs: Any) -> ToolResult:
        if not company:
            return ToolResult(output="", error="company name is required")

        mock_db = {
            "apple": {"name": "Apple Inc.", "industry": "Technology", "founded": 1976, "hq": "Cupertino, CA"},
            "openai": {"name": "OpenAI", "industry": "AI Research", "founded": 2015, "hq": "San Francisco, CA"},
        }
        data = mock_db.get(company.lower())
        if data is None:
            return ToolResult(output="", error=f"No data found for company: {company!r}")

        # Return as JSON so the agent can embed it in a formatted answer
        return ToolResult(output=json.dumps(data))

# Quick test
ci_tool = CompanyInfoTool()
result = asyncio.run(ci_tool.run(company="apple"))
data = json.loads(result.output)
print(data)

## Step 5: Wire tools into an agent

All four tool types plug into any agent class identically — the agent only cares that each object has `name`, `description`, `parameters`, and an async `run()` method.

In [ ]:
from synapsekit.agents import FunctionCallingAgent
from synapsekit.llms.openai import OpenAILLM

agent = FunctionCallingAgent(
    llm=OpenAILLM(model="gpt-4o-mini"),
    tools=[
        celsius_to_fahrenheit,  # @tool decorator instance
        StockPriceTool(),
        ExchangeRateTool(),
        CompanyInfoTool(),
    ],
    system_prompt="You are a helpful financial and unit-conversion assistant.",
)

answer = asyncio.run(agent.run("What is 100 degrees Celsius in Fahrenheit?"))
print(answer)

## Complete Working Example

A self-contained script with a `@tool` decorator tool and a `BaseTool` subclass, wired into a `FunctionCallingAgent` and run against three queries.

In [ ]:
import asyncio
import json
from typing import Any
from synapsekit.agents import BaseTool, FunctionCallingAgent, ToolResult, tool
from synapsekit.llms.openai import OpenAILLM


@tool(name="celsius_to_fahrenheit", description="Convert a Celsius temperature to Fahrenheit.")
def celsius_to_fahrenheit(celsius: float) -> str:
    """Convert Celsius to Fahrenheit."""
    return f"{celsius * 9 / 5 + 32:.1f}\u00b0F"


class StockPriceTool(BaseTool):
    name = "stock_price"
    description = "Return the current stock price for a ticker symbol like AAPL or MSFT."
    parameters = {
        "type": "object",
        "properties": {
            "ticker": {"type": "string", "description": "Uppercase stock ticker, e.g. AAPL"}
        },
        "required": ["ticker"],
    }

    async def run(self, ticker: str = "", **kwargs: Any) -> ToolResult:
        prices = {"AAPL": 189.45, "MSFT": 415.20, "GOOG": 175.30}
        price = prices.get((ticker or "").upper())
        if price is None:
            return ToolResult(output="", error=f"Unknown ticker: {ticker!r}")
        return ToolResult(output=f"{ticker.upper()}: ${price:.2f}")


async def main() -> None:
    agent = FunctionCallingAgent(
        llm=OpenAILLM(model="gpt-4o-mini"),
        tools=[celsius_to_fahrenheit, StockPriceTool()],
        system_prompt="You are a helpful assistant for unit conversion and stock lookups.",
    )

    queries = [
        "What is 22 degrees Celsius in Fahrenheit?",
        "What is the current price of Apple stock?",
        "Convert 37.5\u00b0C to Fahrenheit and also give me the MSFT stock price.",
    ]

    for query in queries:
        print(f"Q: {query}")
        answer = await agent.run(query)
        print(f"A: {answer}\n")


asyncio.run(main())

## Next Steps

- [Multi-Tool Orchestration](https://synapsekit.github.io/synapsekit-docs/docs/guides/agents/multi-tool-orchestration) — compose five or more custom tools in a single agent
- [Tool Error Handling and Retries](https://synapsekit.github.io/synapsekit-docs/docs/guides/agents/tool-error-handling) — build robust retry logic around `ToolResult.is_error`
- [Structured Output with Function Calling](https://synapsekit.github.io/synapsekit-docs/docs/guides/agents/structured-output-function-calling) — return Pydantic models instead of raw strings